# Intercambio de Claves Diffie-Hellman

En este ejercicio vas a implementar el protocolo de intercambio de claves Diffie-Hellman usando aritmética de cuerpos finitos en Sage.

Este protocolo permite que dos partes (Alice y Bob) establezcan un secreto compartido a través de un canal inseguro, sin nunca transmitir el secreto en sí. Es la base de la mayoría de los mecanismos de intercambio de claves usados en la práctica hoy en día.

La seguridad del protocolo se basa en el **Problema del Logaritmo Discreto**: dado $g$ y $g^a$ en un cuerpo finito $\mathbb{F}_p^\times$, es computacionalmente difícil recuperar $a$.

## Calentamiento: Generadores y Logaritmo Discreto

Recordá que $\mathbb{F}_p^\times$ es un grupo cíclico de orden $p - 1$. Un elemento $g$ es un **generador** si todo elemento no nulo de $\mathbb{F}_p$ puede escribirse como $g^k$ para algún entero $k$.

Usaremos el cuerpo de `zk_adventures_types`, donde $p = 65537$.

In [ ]:
from zk_adventures_types import F

p = F.order()
print(f"Working over F_{p}")
print(f"The multiplicative group has order {p - 1}")

### Ejercicio 1: Encontrar un generador

Escribí una función que verifique si un elemento $g \in \mathbb{F}_p$ es un generador de $\mathbb{F}_p^\times$.

**Pista:** Un elemento $g$ es un generador si y solo si su orden multiplicativo es igual a $p - 1$. Sage provee `.multiplicative_order()` sobre elementos de cuerpos finitos.

In [ ]:
def is_generator(g):
    """
    Returns True if `g` is a generator of F_p^×.
    `g` is an element of F (i.e. already wrapped as F(g)).
    """
    raise NotImplementedError("COMPLETE")

In [ ]:
# [TEST]
g = F(3)
assert is_generator(g) == True
assert is_generator(F(1)) == False
assert is_generator(F(p - 1)) == False  # (-1) has order 2
# F.multiplicative_generator() should also be a generator
assert is_generator(F.multiplicative_generator()) == True
print("All generator tests passed!")

### Ejercicio 2: La exponenciación discreta como one-way function

La función $f(x) = g^x \mod p$ es fácil de calcular pero difícil de invertir (ese es el **Problema del Logaritmo Discreto**).

Verifiquemos esto concretamente: calculá $g^a$ y luego usá `discrete_log` de Sage para recuperar $a$.

In [ ]:
g = F(3)
a = 12345

def compute_public_value(generator, secret):
    """
    Given a generator `g` in F and an integer `secret`,
    returns g^secret in F.
    """
    raise NotImplementedError("COMPLETE")

In [ ]:
# [TEST]
A = compute_public_value(g, a)
assert A == g ** a

# Sage can solve discrete log for small fields
recovered_a = discrete_log(A, g)
assert recovered_a == a
print(f"Public value A = g^a = {A}")
print(f"Recovered secret a = {recovered_a}")
print("Discrete exponentiation test passed!")

## El Protocolo Diffie-Hellman

### Descripción del Protocolo

Alice y Bob acuerdan un primo público $p$ y un generador $g$ de $\mathbb{F}_p^\times$.

| Paso | Alice | Bob |
|------|-------|-----|
| 1 | Elige el secreto $a$ | Elige el secreto $b$ |
| 2 | Calcula $A = g^a$ | Calcula $B = g^b$ |
| 3 | Envía $A$ a Bob | Envía $B$ a Alice |
| 4 | Calcula $s = B^a$ | Calcula $s = A^b$ |

Ambos llegan al mismo secreto compartido:
$$s = B^a = (g^b)^a = g^{ab} = (g^a)^b = A^b$$

Un espía ve solo $g$, $A = g^a$ y $B = g^b$, pero no puede calcular eficientemente $g^{ab}$.

### Ejercicio 3: Implementar las partes del protocolo Diffie-Hellman

Completá la clase `DHParty` a continuación. Cada parte:
1. Genera una **clave privada** (un entero aleatorio en $[1, p-2]$).
2. Calcula y publica una **clave pública** $g^{\text{private\_key}}$.
3. Calcula el **secreto compartido** a partir de la clave pública de la otra parte.

In [ ]:
class DHParty:
    def __init__(self, generator):
        """
        Stores the generator.
        Generates a random private key in [1, p-2] where p = F.order().
        Computes the public key as generator^private_key.
        
        Attributes to set:
            self._generator
            self._private_key  (an integer)
            self.public_key    (an element of F)
        """
        raise NotImplementedError("COMPLETE")
    
    def compute_shared_secret(self, other_public_key):
        """
        Given the other party's public key (an element of F),
        computes and returns the shared secret: other_public_key^private_key.
        """
        raise NotImplementedError("COMPLETE")

In [ ]:
# [TEST]
g = F(3)

alice = DHParty(g)
bob = DHParty(g)

# Public keys should be elements of F, not 0 or 1
assert alice.public_key in F
assert bob.public_key in F
assert alice.public_key != F(0)
assert bob.public_key != F(0)

# Both parties compute the same shared secret
alice_secret = alice.compute_shared_secret(bob.public_key)
bob_secret = bob.compute_shared_secret(alice.public_key)
assert alice_secret == bob_secret

# Verify the shared secret equals g^(a*b)
expected = g ** (alice._private_key * bob._private_key)
assert alice_secret == expected

print(f"Alice's public key: {alice.public_key}")
print(f"Bob's public key:   {bob.public_key}")
print(f"Shared secret:      {alice_secret}")
print("Diffie-Hellman key exchange test passed!")

### Ejercicio 4: Versión determinista para testing

Para escribir tests reproducibles, implementá una variante donde la clave privada se provee de forma explícita.

In [ ]:
class DeterministicDHParty:
    def __init__(self, generator, private_key: int):
        """
        Like DHParty, but the private_key is given explicitly.
        Computes the public key as generator^private_key.
        
        Attributes to set:
            self._generator
            self._private_key
            self.public_key
        """
        raise NotImplementedError("COMPLETE")
    
    def compute_shared_secret(self, other_public_key):
        """
        Computes and returns other_public_key^private_key.
        """
        raise NotImplementedError("COMPLETE")

In [ ]:
# [TEST]
g = F(3)
a = 0xCAFE
b = 0xBEEF

alice = DeterministicDHParty(g, a)
bob = DeterministicDHParty(g, b)

assert alice.public_key == g ** a
assert bob.public_key == g ** b

alice_secret = alice.compute_shared_secret(bob.public_key)
bob_secret = bob.compute_shared_secret(alice.public_key)

assert alice_secret == bob_secret
assert alice_secret == g ** (a * b)

# The shared secret should live in F and not be trivial
assert alice_secret in F
assert alice_secret != F(0)
assert alice_secret != F(1)

print(f"Alice public key: g^0xCAFE = {alice.public_key}")
print(f"Bob public key:   g^0xBEEF = {bob.public_key}")
print(f"Shared secret:    g^(0xCAFE * 0xBEEF) = {alice_secret}")
print("Deterministic DH test passed!")

## Variante en Subgrupo: Trabajando en un Subgrupo de Orden Primo

En la práctica, Diffie-Hellman suele realizarse en un **subgrupo de orden primo** de $\mathbb{F}_p^\times$ en lugar del grupo completo. Esto previene ataques de subgrupos pequeños.

Si $p - 1 = q \cdot c$ donde $q$ es un primo grande, podemos encontrar un generador $h$ del subgrupo de orden $q$ calculando $h = g^c$ para cualquier generador $g$ del grupo completo.

Para nuestro cuerpo ($p = 65537$), tenemos $p - 1 = 65536 = 2^{16}$. Como esto es una potencia de 2, el subgrupo de orden primo más grande tiene orden $2$. Para un ejercicio más interesante, trabajemos sobre un cuerpo diferente.

### Ejercicio 5: Diffie-Hellman en un subgrupo de orden primo

Usamos un primo $p$ tal que $p - 1$ tiene un factor primo grande $q$. Las claves privadas deben elegirse en $[1, q-1]$ y todos los cálculos ocurren en el subgrupo de orden $q$.

In [ ]:
# Setup: a safe prime p = 2q + 1 where q is also prime
q = 7879
p_sub = 2 * q + 1  # p_sub = 15759, which is prime
assert is_prime(p_sub)
assert is_prime(q)

F_sub = GF(p_sub)

In [ ]:
def find_subgroup_generator(field, subgroup_order):
    """
    Finds a generator of the subgroup of the given order in field^×.
    
    Strategy: take the field's multiplicative generator `g` and compute
    h = g^((p-1) / subgroup_order), where p = field.order().
    Verify that h has the correct order and h ≠ 1.
    
    Returns h.
    """
    raise NotImplementedError("COMPLETE")

In [ ]:
# [TEST]
h = find_subgroup_generator(F_sub, q)
assert h.multiplicative_order() == q
assert h != F_sub(1)
print(f"Subgroup generator h = {h} with order {h.multiplicative_order()}")

In [ ]:
class SubgroupDHParty:
    def __init__(self, generator, subgroup_order, private_key: int):
        """
        Performs DH in a subgroup of prime order `subgroup_order`.
        
        The private key must satisfy 1 <= private_key < subgroup_order.
        Public key is generator^private_key.
        
        Attributes to set:
            self._generator
            self._subgroup_order
            self._private_key
            self.public_key
        """
        raise NotImplementedError("COMPLETE")
    
    def compute_shared_secret(self, other_public_key):
        """
        Computes the shared secret.
        
        IMPORTANT: Validates that the other party's public key
        is in the correct subgroup (i.e. other_public_key^subgroup_order == 1
        and other_public_key != 1).
        Raises ValueError if validation fails.
        """
        raise NotImplementedError("COMPLETE")

In [ ]:
# [TEST]
h = find_subgroup_generator(F_sub, q)

alice = SubgroupDHParty(h, q, 1234)
bob = SubgroupDHParty(h, q, 5678)

# Public keys are in the subgroup
assert alice.public_key ^ q == F_sub(1)
assert bob.public_key ^ q == F_sub(1)

# Shared secret matches
alice_secret = alice.compute_shared_secret(bob.public_key)
bob_secret = bob.compute_shared_secret(alice.public_key)
assert alice_secret == bob_secret
assert alice_secret == h ^ (1234 * 5678)

# Validation: reject keys not in the subgroup
try:
    alice.compute_shared_secret(F_sub(1))  # identity element
    assert False, "Should have raised ValueError"
except ValueError:
    pass

full_group_gen = F_sub.multiplicative_generator()
assert full_group_gen.multiplicative_order() == p_sub - 1
try:
    alice.compute_shared_secret(full_group_gen)  # not in subgroup of order q
    assert False, "Should have raised ValueError"
except ValueError:
    pass

print("Subgroup DH with validation passed!")

## Ataque de Hombre en el Medio

Diffie-Hellman por sí solo **no** provee autenticación. Un hombre en el medio (Mallory) puede interceptar y reemplazar ambas claves públicas:

| Paso | Alice | Mallory | Bob |
|------|-------|---------|-----|
| 1 | Envía $A = g^a$ | Intercepta $A$ | |
| 2 | | Envía $M_1 = g^{m_1}$ a Bob | Recibe $M_1$ |
| 3 | | | Envía $B = g^b$ |
| 4 | | Intercepta $B$, envía $M_2 = g^{m_2}$ a Alice | |
| 5 | Calcula $s_1 = M_2^{\,a}$ | Conoce $s_1 = A^{m_2}$ y $s_2 = B^{m_1}$ | Calcula $s_2 = M_1^{\,b}$ |

Alice cree que comparte un secreto con Bob, pero en realidad lo comparte con Mallory. Lo mismo le ocurre a Bob.

### Ejercicio 6: Simular el ataque

In [ ]:
def man_in_the_middle(g, alice_secret, bob_secret, mallory_secret_1, mallory_secret_2):
    """
    Simulates a man-in-the-middle attack on Diffie-Hellman.
    
    All secrets are integers. `g` is an element of a finite field.
    
    Returns a dict with:
        'alice_key':     the key Alice computes (thinking it's with Bob)
        'bob_key':       the key Bob computes (thinking it's with Alice)
        'mallory_alice': the key Mallory computes to talk to Alice (should equal alice_key)
        'mallory_bob':   the key Mallory computes to talk to Bob (should equal bob_key)
    """
    raise NotImplementedError("COMPLETE")

In [ ]:
# [TEST]
g = F(3)

result = man_in_the_middle(g,
    alice_secret=111,
    bob_secret=222,
    mallory_secret_1=333,  # Mallory's key with Bob
    mallory_secret_2=444   # Mallory's key with Alice
)

# Mallory shares a key with Alice
assert result['alice_key'] == result['mallory_alice']

# Mallory shares a different key with Bob
assert result['bob_key'] == result['mallory_bob']

# Alice and Bob do NOT share a key with each other
assert result['alice_key'] != result['bob_key']

# Verify concrete values
assert result['alice_key'] == g ^ (111 * 444)
assert result['bob_key'] == g ^ (222 * 333)

print("Alice thinks the shared key is:", result['alice_key'])
print("Bob thinks the shared key is:  ", result['bob_key'])
print("Mallory knows both!")
print("Man-in-the-middle test passed!")

## Mirando hacia Adelante

El protocolo Diffie-Hellman presentado aquí es el bloque fundamental de muchas construcciones criptográficas:

- Las **firmas de Schnorr** (Ejercicio 1 de este curso) usan la misma suposición de logaritmo discreto para construir una prueba de conocimiento.
- El **cifrado ElGamal** extiende DH hacia un esquema de cifrado de clave pública.
- **Diffie-Hellman sobre Curvas Elípticas (ECDH)** reemplaza $\mathbb{F}_p^\times$ por un grupo de curva elíptica para mayor seguridad con claves más pequeñas.

La idea clave siempre es la misma: la exponenciación en un grupo cíclico es una función de una sola vía bajo la suposición del logaritmo discreto.